# NB05 · Adversity + online learning: truth discovery, citizen-report trust, online calibration, drift detection  (CPU · ~10–15 min)

The components that make the system survive bad data and a changing city **without retraining from scratch** (the LightGBM structure is deliberately *not* retrained online).

| Part | Component | Metrics |
|---|---|---|
| A | **CRH-style truth discovery** (reliability-weighted median, EW forgetting 0.98/h) over redundant sensor sources, incl. a *coordinated liar* fraction 0–50 % | consensus MAE vs mean/median, reliability-rank correlation, degradation curve |
| B | **Citizen reports**: Beta-Bernoulli reporter trust + corroboration + burst **quarantine** (coordinated false reports) | precision/recall of false-report flagging, AUROC of report weight, report-rate inflation raw vs trust-weighted |
| C | **Online residual corrector** (2-parameter logistic recalibration of the calibrated logit, SGD, pulled to identity); outcomes fed back only when known (`t+h`). (ACI for interval coverage is exercised in NB04.) | prequential Brier/ECE frozen vs online, before/after drift, no-drift safety check |
| D | **ADWIN** (river if installed, else built-in) on the error stream | detection delay vs injected drift, false alarms on controls |

**Inputs:** NB00 output (required); NB04 `risk_predictions_{test,drift}.csv.gz` for Part C/D (optional).

In [ ]:
import subprocess, sys
try: import river
except Exception: print(subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'river'], capture_output=True, text=True).stdout[-150:])

In [ ]:
import os, sys, glob, json, time, math, warnings, pickle
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore'); plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'figure.figsize': (9, 4)})
ON_KAGGLE = os.path.exists('/kaggle/working')
IN_ROOT = '/kaggle/input' if ON_KAGGLE else os.environ.get('W72_IN', './inputs')
OUT = '/kaggle/working' if ON_KAGGLE else os.environ.get('W72_OUT', './outputs')
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
def find(name, required=True):
    hits = sorted(glob.glob(f'{IN_ROOT}/**/{name}', recursive=True) + glob.glob(f'{OUT}/**/{name}', recursive=True))
    if not hits and required: raise FileNotFoundError(f'{name} not found under {IN_ROOT} - add the producing notebook/dataset via "Add Input"')
    return hits[0] if hits else None
def savefig(n): plt.tight_layout(); plt.savefig(f'{FIG}/{n}.png', bbox_inches='tight'); plt.show()
sys.path.insert(0, os.path.dirname(find('w72_common.py'))); import w72_common as W

In [ ]:
QUICK = False
CFG = dict(n_eps=12, n_rep=10)
if QUICK: CFG.update(n_eps=3, n_rep=2)
MAN = pd.read_csv(find('w72_manifest.csv')); ROOT = os.path.dirname(find('w72_manifest.csv')); rng = np.random.default_rng(0)
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, brier_score_loss

## A · Truth discovery over redundant sources

In [ ]:
def wmedian(v, w):
    ok = ~np.isnan(v); v, w = v[ok], w[ok]
    if len(v) == 0: return np.nan
    o = np.argsort(v); cw = np.cumsum(w[o]); return v[o][np.searchsorted(cw, 0.5 * cw[-1])]
SIG = np.array([0.01, 0.02, 0.03, 0.06, 0.15, 0.30])          # true per-source noise (fraction of variable range) -- hidden from the algorithm
def make_claims(x, n_bad, rg, rng_v):
    K = len(SIG); bad = rg.choice(K, n_bad, replace=False) if n_bad else np.array([], int); C = x[None] + rg.normal(0, 1, (K,) + x.shape) * (SIG[:, None, None] * rng_v)
    C[rg.random(C.shape) < 0.1] = np.nan                                                                          # 10 % dropout everywhere
    for b in bad: C[b] = 0.3 * x + 0.05 * rng_v + rg.normal(0, 0.01 * rng_v, x.shape)                          # COORDINATED liars share one story
    return C, bad
def crh(C, rng_v, decay=0.98, iters=3):
    K, T, Z = C.shape; L = np.full(K, 0.05); R = np.ones(K); cons = np.zeros((T, Z)); Rh = np.zeros((T, K))
    for t in range(T):
        for _ in range(iters):
            cons[t] = [wmedian(C[:, t, z], R) for z in range(Z)]; loss = np.nanmean(np.abs(C[:, t, :] - cons[t][None]) / rng_v, axis=1); loss = np.where(np.isnan(loss), np.nanmean(loss), loss)
            Lt = decay * L + (1 - decay) * loss; R = np.clip(np.exp(-Lt / (2 * np.median(Lt) + 1e-4)), 0.05, 1)
        L = Lt; Rh[t] = R
    return cons, Rh
rows = []; traj = None; eps_ = MAN[(MAN.split == 'test') & MAN.hazard.isin(['flood', 'compoundA'])].head(CFG['n_eps'])
if len(eps_) == 0: eps_ = MAN[MAN.split == 'test'].head(CFG['n_eps'])
for nb in tqdm([0, 1, 2, 3], desc='corrupted sources'):
    for r in eps_.itertuples():
        d = W.load_episode(f'{ROOT}/{r.path}'); v = W.VI['water_depth_cm']; x = d['X'][48:168, :, v]; rv = W.VAR_MAX[v] - W.VAR_MIN[v]; rg = np.random.default_rng(int(r.seed) + nb)
        C, bad = make_claims(x, nb, rg, rv); cons, Rh = crh(C, rv); mean_ = np.nanmean(C, 0); med_ = np.nanmedian(C, 0)
        good = [k for k in range(6) if k not in bad]; rk = spearmanr(Rh[-1][good], -SIG[good])[0] if len(good) > 2 else np.nan
        rows.append(dict(n_bad=nb, pct_bad=100 * nb / 6, mae_mean=np.abs(mean_ - x).mean(), mae_median=np.abs(med_ - x).mean(), mae_crh=np.abs(cons - x).mean(), rank_corr=rk, w_liars=Rh[-1][bad].mean() if nb else np.nan, w_good=Rh[-1][good].mean()))
        if nb == 2 and traj is None: traj = (Rh, bad)
TD = pd.DataFrame(rows).groupby('pct_bad').mean(numeric_only=True); display(TD.round(4))
fig, ax = plt.subplots(1, 3, figsize=(17, 4)); [ax[0].plot(TD.index, TD[c], 'o-', label=c) for c in ['mae_mean', 'mae_median', 'mae_crh']]; ax[0].set_xlabel('% sources that are coordinated liars'); ax[0].set_ylabel('MAE (water depth, cm)'); ax[0].legend(); ax[0].set_title('consensus error vs corruption')
Rh, bad = traj; [ax[1].plot(Rh[:, k], label=f'src{k} σ={SIG[k]}' + (' LIAR' if k in bad else ''), lw=2 if k in bad else 1) for k in range(6)]; ax[1].set_title('learned reliability over time (2 liars)'); ax[1].legend(fontsize=6); ax[1].set_xlabel('hour')
ax[2].bar(['good', 'liars'], [TD.w_good.loc[33.33333333333333] if 33.33333333333333 in TD.index else TD.w_good.iloc[-2], TD.w_liars.loc[33.33333333333333] if 33.33333333333333 in TD.index else TD.w_liars.iloc[-2]], color=['tab:green', 'tab:red']); ax[2].set_title('final weight, 2 of 6 liars'); savefig('50_truth_discovery')

## B · Citizen reports: trust, corroboration, quarantine

In [ ]:
def simulate_reports(d, rg, n_honest=400, n_spam=30, win=18):
    S = W.stress_from_vars(d['X']); sev = np.maximum(S[..., W.LI['HYD']], S[..., W.LI['TRF']]); T, Z = sev.shape; sens = sev + rg.normal(0, 0.05, sev.shape)        # sensor-derived severity
    rows = []; zf = int(np.argmin(sev.mean(0))); t0 = int(rg.integers(60, 150))
    for t in range(48, 200):
        for z in range(Z):
            n = rg.poisson(0.05 + 2.0 * np.clip(sev[t, z] - 0.2, 0, 1))
            for _ in range(n): rows.append((t, z, int(rg.integers(n_honest)), 0))
            if rg.random() < 0.01: rows.append((t, z, int(rg.integers(n_honest)), 1))                                        # honest-but-wrong report (noise)
        if t0 <= t < t0 + win:
            for _ in range(40): rows.append((t, zf, n_honest + int(rg.integers(n_spam)), 1))                                   # coordinated spam burst in a calm zone
    return pd.DataFrame(rows, columns=['t', 'zone', 'rep', 'false']), sens, zf, t0
def trust_pipeline(R, sens, thr_corr=0.35):
    a = {}; b = {}; seen = {}; w = np.zeros(len(R)); flag = np.zeros(len(R), bool); R = R.reset_index(drop=True)
    for (t, z), g in R.groupby(['t', 'zone'], sort=True):
        sensor_ok = sens[t, z] > thr_corr; n_ind = g.rep.nunique(); new_frac = np.mean([seen.get(r_, 0) < 5 for r_ in g.rep]); burst = (len(g) >= 10) and (new_frac >= 0.6) and (not sensor_ok)
        for i, r_ in zip(g.index, g.rep):
            tr = (1 + a.get(r_, 0)) / (2 + a.get(r_, 0) + b.get(r_, 0)); corr = 1 - 0.5 ** (n_ind - 1 + sensor_ok); w[i] = 0 if burst else tr * (0.25 + 0.75 * corr); flag[i] = burst or w[i] < 0.12
        for i, r_ in zip(g.index, g.rep):
            if burst or (not sensor_ok and len(g) >= 3): b[r_] = b.get(r_, 0) + 1
            elif sensor_ok or n_ind >= 2: a[r_] = a.get(r_, 0) + 1
            seen[r_] = seen.get(r_, 0) + 1
    return w, flag
rows = []; ex = None
for r in tqdm(list(MAN[MAN.split == 'test'].head(CFG['n_rep']).itertuples()), desc='report episodes'):
    d = W.load_episode(f'{ROOT}/{r.path}'); rg = np.random.default_rng(int(r.seed)); R, sens, zf, t0 = simulate_reports(d, rg); w, flag = trust_pipeline(R, sens); y = R['false'].values
    m = ((R.t >= t0) & (R.t < t0 + 18) & (R.zone == zf)).values          # reports/hour in the ATTACKED calm zone during the burst
    rate_raw, rate_trust, rate_honest = m.sum() / 18, w[m].sum() / 18, (m & (y == 0)).sum() / 18
    rows.append(dict(ep=r.ep_id, n_reports=len(R), false_share=y.mean(), auroc_weight=roc_auc_score(1 - y, w) if 0 < y.mean() < 1 else np.nan, precision=(y[flag].mean() if flag.any() else np.nan), recall=flag[y == 1].mean() if (y == 1).any() else np.nan,
                     honest_flagged=flag[y == 0].mean(), spam_weight=w[y == 1].mean(), honest_weight=w[y == 0].mean(), rate_raw=rate_raw, rate_trust_weighted=rate_trust, rate_honest_truth=rate_honest))
    if ex is None: ex = (R, w, flag, zf, t0)
RP = pd.DataFrame(rows); display(RP.round(3)); print(RP.mean(numeric_only=True).round(3).to_dict())
R, w, flag, zf, t0 = ex; fig, ax = plt.subplots(1, 3, figsize=(17, 4)); ax[0].hist([w[R['false'].values == 0], w[R['false'].values == 1]], bins=20, label=['honest', 'false/spam'], color=['tab:green', 'tab:red']); ax[0].legend(); ax[0].set_title('report weight distribution'); ax[0].set_yscale('log')
ct = R.assign(flag=flag, w=w); a_ = ct[ct.zone == zf].groupby('t').agg(raw=('t', 'size'), weighted=('w', 'sum'), quarantined=('flag', 'sum')); a_.plot(ax=ax[1]); ax[1].axvspan(t0, t0 + 18, color='red', alpha=.1); ax[1].set_title(f'false zone {zf}: raw count vs trust-weighted vs quarantined')
RP[['rate_raw', 'rate_trust_weighted', 'rate_honest_truth']].mean().plot.bar(ax=ax[2], color=['tab:red', 'tab:green', 'gray']); ax[2].set_yscale('log'); ax[2].set_title('reports/hour in the attacked zone: raw vs trust-weighted vs honest truth'); savefig('51_citizen_trust')

## C · Online residual corrector + ACI  &  D · ADWIN drift detection (consumes NB04 prediction logs)

In [ ]:
pt = find('risk_predictions_drift.csv.gz', False); ptest = find('risk_predictions_test.csv.gz', False)
HAVE = pt is not None
if HAVE:
    DR = pd.read_csv(pt); TE = pd.read_csv(ptest) if ptest else None; DR = DR[DR.h.isin([6, 24, 72])]; print('drift rows', len(DR), 'episodes', DR.ep.nunique())
    if TE is not None: TE = TE[TE.h.isin([6, 24, 72])]
else: print('NB04 prediction logs not attached: skipping C/D')

In [ ]:
lg_ = lambda p: np.log(np.clip(p, 1e-5, 1 - 1e-5) / (1 - np.clip(p, 1e-5, 1 - 1e-5)))
class OnlineCorrector:
    # p_online = sigmoid(slope*logit(p_cal) + intercept); init = identity; SGD on log-loss with L2 pull to identity (2 parameters: cannot run away)
    def __init__(s, lr=0.002, l2=0.05): s.w = np.array([1.0, 0.0]); s.w0 = s.w.copy(); s.lr = lr; s.l2 = l2
    def f(s, z, h): return np.array([z, 1.0])
    def predict(s, z, h): return 1 / (1 + np.exp(-s.w @ s.f(z, h)))
    def learn(s, z, h, y): x = s.f(z, h); p = 1 / (1 + np.exp(-s.w @ x)); s.w -= s.lr * ((p - y) * x + s.l2 * (s.w - s.w0))
class Detector:
    def __init__(s):
        try: from river import drift; s.d = drift.ADWIN(delta=0.002); s.kind = 'river.ADWIN'
        except Exception: s.d = None; s.kind = 'built-in Page-Hinkley'; s.n = 0; s.m = 0.0; s.cum = 0.0; s.mn = 0.0
    def update(s, v):
        if s.d is not None: s.d.update(v); return bool(s.d.drift_detected)
        s.n += 1; s.m += (v - s.m) / s.n; s.cum += v - s.m; s.mx = max(getattr(s, 'mx', 0.0), s.cum); s.mn = min(s.mn, s.cum)
        if s.n > 200 and max(s.cum - s.mn, s.mx - s.cum) > 8.0: s.n = 0; s.m = s.cum = s.mn = s.mx = 0.0; return True
        return False
def run_episode(g, drift_at, lr=0.002):
    g = g.sort_values(['t', 'zone', 'h']); ev = []
    for r in g.itertuples(): ev.append((r.t, 0, r))                                       # predict at t
    for r in g.itertuples(): ev.append((r.t + r.h, 1, r))                                 # outcome known at t+h
    ev.sort(key=lambda e: (e[0], e[1])); oc = OnlineCorrector(lr); det = Detector(); out = []; alarm = None
    pend = {}
    for tm, kind, r in ev:
        z = lg_(r.p_cal); key = (r.t, r.zone, r.h)
        if kind == 0: pend[key] = oc.predict(z, r.h); out.append((r.t, r.h, r.p_cal, pend[key], r.y))
        else:
            oc.learn(z, r.h, r.y); err = r.y - r.p_cal                      # signed calibration residual: a mean shift = calibration drift
            if det.update(err) and alarm is None and tm >= 52: alarm = tm
    return pd.DataFrame(out, columns=['t', 'h', 'p_frozen', 'p_online', 'y']), alarm, oc.w, det.kind
def ece_(p, y, nb=10): o = np.argsort(p); b = np.array_split(o, nb); return float(sum(len(x) * abs(p[x].mean() - y[x].mean()) for x in b) / len(p))
if HAVE:
    res = []; alarms = {}; wts = []
    for ep, g in tqdm(DR.groupby('ep'), desc='stream drift episodes'):
        is_dr = int(ep.split('_')[1]) < DR.ep.nunique() // 2; o, alarm, w_, kind = run_episode(g, 100 if is_dr else None); alarms[ep] = (is_dr, alarm); wts.append(w_)
        for nm, m in [('pre', (o.t >= 60) & (o.t < 100)), ('post', (o.t >= 100) & (o.t < 170))]:
            if m.sum() > 50: res.append(dict(ep=ep, drifted=is_dr, window=nm, brier_frozen=brier_score_loss(o.y[m], o.p_frozen[m]), brier_online=brier_score_loss(o.y[m], o.p_online[m]), ece_frozen=ece_(o.p_frozen[m].values, o.y[m].values), ece_online=ece_(o.p_online[m].values, o.y[m].values)))
    RS = pd.DataFrame(res); S_ = RS.groupby(['drifted', 'window']).mean(numeric_only=True); display(S_.round(5)); print('detector:', kind)
    dd = [a for e, (d_, a) in alarms.items() if d_ and a is not None]; cc = [a for e, (d_, a) in alarms.items() if (not d_) and a is not None]; nd = sum(d_ for d_, _ in alarms.values()); nc = len(alarms) - nd
    print(f'alarms: drifted {len(dd)}/{nd} (median alarm time {np.median(dd) if dd else None}, injected at t=100) | control {len(cc)}/{nc}')

In [ ]:
if HAVE:
    # no-drift safety check on the (mixed-hazard, mixed-fault) TEST episodes: online correction must not hurt
    safe = []
    if TE is not None:
        for ep, g in tqdm(list(TE.groupby('ep'))[:CFG['n_eps'] * 2], desc='stream test episodes'):
            o, _, _, _ = run_episode(g, None); m = (o.t >= 60)
            if m.sum() > 50 and 0 < o.y[m].mean() < 1: safe.append(dict(ep=ep, brier_frozen=brier_score_loss(o.y[m], o.p_frozen[m]), brier_online=brier_score_loss(o.y[m], o.p_online[m]), ece_frozen=ece_(o.p_frozen[m].values, o.y[m].values), ece_online=ece_(o.p_online[m].values, o.y[m].values)))
    SF = pd.DataFrame(safe); print('NO-DRIFT test episodes (mean):', SF.mean(numeric_only=True).round(5).to_dict() if len(SF) else 'n/a')
    fig, ax = plt.subplots(1, 3, figsize=(17, 4)); P_ = S_.reset_index(); lab = P_.drifted.map({True: 'drifted', False: 'control'}) + ' / ' + P_.window
    x = np.arange(len(P_)); ax[0].bar(x - .2, P_.brier_frozen, .4, label='frozen'); ax[0].bar(x + .2, P_.brier_online, .4, label='online'); ax[0].set_xticks(x); ax[0].set_xticklabels(lab, rotation=20); ax[0].set_title('Brier (lower is better)'); ax[0].legend()
    ax[1].bar(x - .2, P_.ece_frozen, .4, label='frozen'); ax[1].bar(x + .2, P_.ece_online, .4, label='online'); ax[1].set_xticks(x); ax[1].set_xticklabels(lab, rotation=20); ax[1].set_title('ECE'); ax[1].legend()
    ax[2].hist([dd, cc] if (dd or cc) else [[0]], bins=10, label=['drifted', 'control']); ax[2].axvline(100, c='r', ls='--'); ax[2].set_title(f'{kind} alarm times (drift injected at t=100)'); ax[2].legend(); savefig('52_online_calibration_drift')
    json.dump(dict(truth_discovery=TD.reset_index().to_dict('records'), citizen=RP.mean(numeric_only=True).to_dict(), online_drift=S_.reset_index().to_dict('records'), alarms=dict(drifted=f'{len(dd)}/{nd}', control=f'{len(cc)}/{nc}'), detector=kind,
                   online_weights_mean=np.mean(wts, 0).tolist(), nodrift_check=SF.mean(numeric_only=True).to_dict() if len(SF) else None), open(f'{OUT}/adversity_online_report.json', 'w'), indent=1, default=float)
else: json.dump(dict(truth_discovery=TD.reset_index().to_dict('records'), citizen=RP.mean(numeric_only=True).to_dict()), open(f'{OUT}/adversity_online_report.json', 'w'), indent=1, default=float)
print('saved adversity_online_report.json')